# RSNA Knee Abnormality Detection — Exploratory Data Analysis

This notebook profiles the competition's **study-level labels and reports**, **series-level acquisition metadata**, and a small, bounded sample of **DICOM headers and pixels**. It is deliberately metadata-first: the full image corpus is about 570 GB, so recursively loading every image is neither necessary nor practical for EDA.

The analysis is designed to run on Kaggle with the competition data attached. All labels are treated as partially observed: a blank target is **unknown**, not negative.

## Questions

1. How complete and imbalanced are the twelve targets?
2. Which abnormalities tend to co-occur?
3. What demographic, report, and acquisition variation is present?
4. Are study-to-series relationships and DICOM files internally consistent?
5. What do these observations imply for validation, preprocessing, and modeling?

In [ ]:
from pathlib import Path
from collections import Counter
import math
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (11, 5)
plt.rcParams["axes.titlesize"] = 14

SEED = 42
LABEL_COLS = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus",
    "Medial OA", "Lateral OA", "PF OA", "Effusion",
    "Synovitis", "Baker's", "Contusion", "Fracture",
]
ID_COL = "StudyInstanceUID"
SERIES_ID_COL = "SeriesInstanceUID"

## 1. Locate and load the tabular data

The path resolver supports Kaggle's attached-input layout and a local checkout with CSVs in `data/`, `input/`, or the repository root. Set `DATA_DIR` manually if auto-detection is ambiguous.

In [ ]:
def find_data_dir():
    candidates = [
        Path("/kaggle/input/rsna-knee-abnormality-detection"),
        Path("/kaggle/input/rsna-knee-abnormalities-detection"),
        Path("../input/rsna-knee-abnormality-detection"),
        Path("../input/rsna-knee-abnormalities-detection"),
        Path("data"), Path("input"), Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "train_series.csv").exists():
            return path.resolve()
    kaggle_root = Path("/kaggle/input")
    if kaggle_root.exists():
        for train_path in kaggle_root.glob("*/train.csv"):
            if (train_path.parent / "train_series.csv").exists():
                return train_path.parent.resolve()
    raise FileNotFoundError(
        "Could not locate train.csv and train_series.csv. Attach the competition data "
        "or set DATA_DIR to the directory containing the CSV files."
    )

DATA_DIR = find_data_dir()
print(f"Using data from: {DATA_DIR}")

train = pd.read_csv(DATA_DIR / "train.csv")
train_series = pd.read_csv(DATA_DIR / "train_series.csv")
test = pd.read_csv(DATA_DIR / "test.csv") if (DATA_DIR / "test.csv").exists() else None
test_series = pd.read_csv(DATA_DIR / "test_series.csv") if (DATA_DIR / "test_series.csv").exists() else None
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv") if (DATA_DIR / "sample_submission.csv").exists() else None

print(f"train:        {train.shape}")
print(f"train_series: {train_series.shape}")
if test is not None: print(f"test:         {test.shape}")
if test_series is not None: print(f"test_series:  {test_series.shape}")
display(train.head(3))
display(train_series.head(3))

## 2. Schema and integrity checks

In [ ]:
expected_train = {ID_COL, "PatientSex", "Report", *LABEL_COLS}
expected_series = {ID_COL, SERIES_ID_COL, "Fluid_Sensitive", "Fat_Suppression", "Anatomical_Plane"}

schema_checks = pd.Series({
    "train has expected columns": expected_train.issubset(train.columns),
    "train_series has expected columns": expected_series.issubset(train_series.columns),
    "study IDs unique in train": train[ID_COL].is_unique,
    "series IDs unique": train_series[SERIES_ID_COL].is_unique,
    "no missing study IDs": train[ID_COL].notna().all() and train_series[ID_COL].notna().all(),
    "all series map to a train study": train_series[ID_COL].isin(train[ID_COL]).all(),
})
display(schema_checks.rename("passed").to_frame())

orphan_series = train_series.loc[~train_series[ID_COL].isin(train[ID_COL])]
studies_without_series = train.loc[~train[ID_COL].isin(train_series[ID_COL]), ID_COL]
print(f"Duplicate train study IDs: {train[ID_COL].duplicated().sum():,}")
print(f"Duplicate series IDs:      {train_series[SERIES_ID_COL].duplicated().sum():,}")
print(f"Orphan series rows:        {len(orphan_series):,}")
print(f"Studies without a series:  {len(studies_without_series):,}")

for col in LABEL_COLS:
    values = set(train[col].dropna().unique())
    assert values <= {0, 1, 0.0, 1.0}, f"Unexpected values in {col}: {values}"

In [ ]:
def dataframe_summary(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "non_null": df.notna().sum(),
        "missing": df.isna().sum(),
        "missing_pct": (100 * df.isna().mean()).round(2),
        "unique": df.nunique(dropna=True),
    }).sort_values(["missing_pct", "unique"], ascending=[False, True])

display(dataframe_summary(train))
display(dataframe_summary(train_series))

## 3. Label availability and prevalence

Because only a subset of studies is labeled for each condition, two rates matter:

- **coverage**: fraction of all studies with an observed label;
- **prevalence**: fraction positive among observed labels only.

This distinction must also be preserved in training: missing targets require a masked loss.

In [ ]:
label_summary = pd.DataFrame(index=LABEL_COLS)
label_summary["labeled"] = train[LABEL_COLS].notna().sum()
label_summary["missing"] = train[LABEL_COLS].isna().sum()
label_summary["coverage_pct"] = 100 * label_summary["labeled"] / len(train)
label_summary["positive"] = train[LABEL_COLS].eq(1).sum()
label_summary["negative"] = train[LABEL_COLS].eq(0).sum()
label_summary["prevalence_pct"] = 100 * label_summary["positive"] / label_summary["labeled"]
label_summary["neg_pos_ratio"] = label_summary["negative"] / label_summary["positive"].replace(0, np.nan)
label_summary = label_summary.round(2).sort_values("prevalence_pct", ascending=False)
display(label_summary)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sns.barplot(data=label_summary.reset_index(), y="index", x="coverage_pct", ax=axes[0], color="#4C78A8")
axes[0].set(title="Observed-label coverage", xlabel="Studies labeled (%)", ylabel="")
axes[0].set_xlim(0, 100)
sns.barplot(data=label_summary.reset_index(), y="index", x="prevalence_pct", ax=axes[1], color="#E45756")
axes[1].set(title="Positive prevalence among labeled studies", xlabel="Positive (%)", ylabel="")
plt.tight_layout()
plt.show()

In [ ]:
observed_per_study = train[LABEL_COLS].notna().sum(axis=1)
positive_per_study = train[LABEL_COLS].eq(1).sum(axis=1)

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.countplot(x=observed_per_study, color="#72B7B2", ax=axes[0])
axes[0].set(title="Observed targets per study", xlabel="Number of observed targets", ylabel="Studies")
sns.countplot(x=positive_per_study, color="#F58518", ax=axes[1])
axes[1].set(title="Positive targets per study", xlabel="Number of positives", ylabel="Studies")
plt.tight_layout()
plt.show()

print(f"Studies with no observed target: {(observed_per_study == 0).sum():,}")
print(f"Studies with all 12 targets observed: {(observed_per_study == len(LABEL_COLS)).sum():,}")
print(f"Studies with at least one observed positive: {(positive_per_study > 0).sum():,}")

## 4. Co-occurrence and target relationships

In [ ]:
# Pairwise-complete Pearson correlation (the phi coefficient for binary pairs).
label_corr = train[LABEL_COLS].corr(min_periods=30)
mask = np.triu(np.ones_like(label_corr, dtype=bool), k=1)
plt.figure(figsize=(11, 9))
sns.heatmap(label_corr, mask=mask, cmap="vlag", center=0, vmin=-1, vmax=1,
            annot=True, fmt=".2f", square=True, cbar_kws={"shrink": .75})
plt.title("Target correlation (pairwise observed labels)")
plt.tight_layout()
plt.show()

pairs = []
for i, left in enumerate(LABEL_COLS):
    for right in LABEL_COLS[i + 1:]:
        valid = train[[left, right]].dropna()
        both_positive = ((valid[left] == 1) & (valid[right] == 1)).sum()
        pairs.append({
            "label_1": left, "label_2": right, "jointly_labeled": len(valid),
            "both_positive": both_positive,
            "co_positive_pct": 100 * both_positive / len(valid) if len(valid) else np.nan,
            "correlation": valid[left].corr(valid[right]) if len(valid) > 1 else np.nan,
        })
pair_summary = pd.DataFrame(pairs)
display(pair_summary.sort_values("correlation", ascending=False).head(12).round(3))

## 5. Patient sex and label prevalence

In [ ]:
sex = train["PatientSex"].fillna("Missing").replace("", "Missing")
display(sex.value_counts(dropna=False).rename_axis("PatientSex").to_frame("studies"))

plt.figure(figsize=(7, 4))
sns.countplot(x=sex, order=sex.value_counts().index, color="#54A24B")
plt.title("Patient sex distribution")
plt.xlabel("PatientSex")
plt.ylabel("Studies")
plt.tight_layout()
plt.show()

sex_prev = train.assign(PatientSexClean=sex).groupby("PatientSexClean")[LABEL_COLS].mean().T * 100
sex_counts = train.assign(PatientSexClean=sex).groupby("PatientSexClean")[LABEL_COLS].count().T
display(sex_prev.round(2).add_suffix(" prevalence_pct"))
display(sex_counts.add_suffix(" labeled_n"))

## 6. Radiology-report characteristics

Reports are multimodal inputs and potential sources of weak supervision. This section measures availability, length, common tokens, and rough script usage without sending text outside the notebook environment. Token counts are descriptive—not a clinical NLP pipeline.

In [ ]:
reports = train["Report"].fillna("").astype(str)
report_stats = pd.DataFrame({
    "characters": reports.str.len(),
    "words": reports.str.findall(r"\b\w+\b").str.len(),
    "lines": reports.str.count(r"\n") + reports.ne("").astype(int),
})
print(f"Missing/empty reports: {(reports.str.strip() == '').sum():,} ({100*(reports.str.strip() == '').mean():.2f}%)")
display(report_stats.describe(percentiles=[.25, .5, .75, .9, .95, .99]).round(1))

fig, axes = plt.subplots(1, 2, figsize=(15, 4))
sns.histplot(report_stats.loc[report_stats.words > 0, "words"], bins=60, ax=axes[0], color="#B279A2")
axes[0].set(title="Report length", xlabel="Words")
sns.histplot(np.log1p(report_stats.loc[report_stats.words > 0, "words"]), bins=50, ax=axes[1], color="#FF9DA6")
axes[1].set(title="Report length (log scale)", xlabel="log(1 + words)")
plt.tight_layout()
plt.show()

In [ ]:
def script_flags(text):
    return {
        "Latin": bool(re.search(r"[A-Za-z]", text)),
        "Cyrillic": bool(re.search(r"[\u0400-\u04FF]", text)),
        "CJK": bool(re.search(r"[\u3400-\u9FFF]", text)),
        "Arabic": bool(re.search(r"[\u0600-\u06FF]", text)),
        "Devanagari": bool(re.search(r"[\u0900-\u097F]", text)),
    }

script_counts = pd.DataFrame(reports.map(script_flags).tolist()).sum().sort_values(ascending=False)
display(script_counts.rename("reports_containing_script").to_frame())

STOPWORDS = {
    "the", "and", "of", "a", "to", "in", "is", "with", "for", "on", "no", "are",
    "there", "this", "that", "at", "from", "or", "an", "be", "was", "as", "it",
}
tokens = Counter()
for text in reports:
    tokens.update(t for t in re.findall(r"[A-Za-z]{3,}", text.lower()) if t not in STOPWORDS)
display(pd.DataFrame(tokens.most_common(30), columns=["token", "count"]))

### Report leakage check

The report is an allowed input, but explicit diagnosis terms may make the task close to label extraction for some targets. The table below is only a rough lexical audit; it does not establish clinical negation or temporality.

In [ ]:
keywords = {
    "ACL": r"\b(?:acl|anterior cruciate)\b",
    "MCL": r"\b(?:mcl|medial collateral)\b",
    "Medial Meniscus": r"\bmedial menisc\w*\b",
    "Lateral Meniscus": r"\blateral menisc\w*\b",
    "Medial OA": r"\b(?:medial.*(?:osteoarthritis|arthrosis)|osteoarthritis.*medial)\b",
    "Lateral OA": r"\b(?:lateral.*(?:osteoarthritis|arthrosis)|osteoarthritis.*lateral)\b",
    "PF OA": r"\b(?:patellofemoral.*(?:osteoarthritis|arthrosis)|(?:osteoarthritis|arthrosis).*patellofemoral)\b",
    "Effusion": r"\beffusion\b",
    "Synovitis": r"\bsynovitis\b",
    "Baker's": r"\b(?:baker'?s?|popliteal) cyst\b",
    "Contusion": r"\b(?:bone )?(?:contusion|bruise)\b",
    "Fracture": r"\bfractur\w*\b",
}

lexical_rows = []
for label, pattern in keywords.items():
    mention = reports.str.contains(pattern, case=False, regex=True, na=False)
    valid = train[label].notna()
    pos = train[label].eq(1) & valid
    neg = train[label].eq(0) & valid
    lexical_rows.append({
        "label": label,
        "reports_with_keyword": int(mention.sum()),
        "keyword_rate_in_positive_pct": 100 * (mention & pos).sum() / max(pos.sum(), 1),
        "keyword_rate_in_negative_pct": 100 * (mention & neg).sum() / max(neg.sum(), 1),
    })
display(pd.DataFrame(lexical_rows).set_index("label").round(2))

## 7. MRI series composition

In [ ]:
series_per_study = train_series.groupby(ID_COL).size().rename("series_count")
display(series_per_study.describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).round(2).to_frame())

fig, axes = plt.subplots(1, 3, figsize=(18, 4))
sns.histplot(series_per_study, discrete=True, ax=axes[0], color="#4C78A8")
axes[0].set(title="Series per study", xlabel="Series count")
sns.countplot(data=train_series, x="Anatomical_Plane",
              order=train_series["Anatomical_Plane"].value_counts().index, ax=axes[1], color="#F58518")
axes[1].set(title="Anatomical plane", xlabel="")
for col, color in [("Fluid_Sensitive", "#54A24B"), ("Fat_Suppression", "#E45756")]:
    train_series[col].value_counts(normalize=True).sort_index().plot(kind="bar", alpha=.7, ax=axes[2], label=col, color=color)
axes[2].set(title="Sequence flags", xlabel="Flag value", ylabel="Fraction of series")
axes[2].legend()
plt.tight_layout()
plt.show()

In [ ]:
protocol_counts = (
    train_series.assign(
        Fluid_Sensitive=train_series["Fluid_Sensitive"].fillna("Missing").astype(str),
        Fat_Suppression=train_series["Fat_Suppression"].fillna("Missing").astype(str),
        Anatomical_Plane=train_series["Anatomical_Plane"].fillna("Missing"),
    )
    .groupby(["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"], dropna=False)
    .size().rename("series").reset_index().sort_values("series", ascending=False)
)
display(protocol_counts.head(20))

plane_presence = pd.crosstab(train_series[ID_COL], train_series["Anatomical_Plane"]).gt(0)
for plane in ["Sagittal", "Coronal", "Axial"]:
    if plane not in plane_presence: plane_presence[plane] = False
plane_presence["all_three_planes"] = plane_presence[["Sagittal", "Coronal", "Axial"]].all(axis=1)
display((100 * plane_presence.mean()).round(2).rename("studies_pct").to_frame())

## 8. Bounded DICOM audit

Only a reproducible sample of series is inspected. Header reads use `stop_before_pixels=True`; pixel arrays are decoded for at most 12 representative files. This catches common variation in matrix size, spacing, thickness, transfer syntax, and intensity without scanning the entire image corpus.

In [ ]:
try:
    import pydicom
    PYDICOM_AVAILABLE = True
except ImportError:
    PYDICOM_AVAILABLE = False
    print("pydicom is unavailable; skipping DICOM audit.")

TRAIN_IMAGE_DIR = DATA_DIR / "train_series"
SAMPLED_SERIES = 60

def dicom_paths_for_series(row):
    folder = TRAIN_IMAGE_DIR / str(row[ID_COL]) / str(row[SERIES_ID_COL])
    return sorted(folder.glob("*.dcm")) if folder.exists() else []

dicom_rows = []
sampled_files = []
if PYDICOM_AVAILABLE and TRAIN_IMAGE_DIR.exists():
    sampled_meta = train_series.sample(min(SAMPLED_SERIES, len(train_series)), random_state=SEED)
    for _, row in sampled_meta.iterrows():
        files = dicom_paths_for_series(row)
        if not files:
            continue
        sampled_files.append(files[len(files) // 2])
        ds = pydicom.dcmread(files[len(files) // 2], stop_before_pixels=True, force=True)
        spacing = getattr(ds, "PixelSpacing", [np.nan, np.nan])
        dicom_rows.append({
            ID_COL: row[ID_COL], SERIES_ID_COL: row[SERIES_ID_COL],
            "plane": row.get("Anatomical_Plane"), "slices": len(files),
            "rows": getattr(ds, "Rows", np.nan), "columns": getattr(ds, "Columns", np.nan),
            "spacing_row": float(spacing[0]) if len(spacing) > 0 else np.nan,
            "spacing_col": float(spacing[1]) if len(spacing) > 1 else np.nan,
            "slice_thickness": float(getattr(ds, "SliceThickness", np.nan)),
            "bits_stored": getattr(ds, "BitsStored", np.nan),
            "transfer_syntax": str(getattr(getattr(ds, "file_meta", None), "TransferSyntaxUID", "Unknown")),
        })

dicom_sample = pd.DataFrame(dicom_rows)
if not dicom_sample.empty:
    display(dicom_sample.describe(include="all").T)
    display(dicom_sample["transfer_syntax"].value_counts().to_frame("sampled_files"))

In [ ]:
if PYDICOM_AVAILABLE and sampled_files:
    pixel_files = sampled_files[:12]
    fig, axes = plt.subplots(3, 4, figsize=(13, 11))
    for ax, path in zip(axes.flat, pixel_files):
        try:
            ds = pydicom.dcmread(path, force=True)
            image = ds.pixel_array.astype(np.float32)
            lo, hi = np.percentile(image, [1, 99])
            ax.imshow(image, cmap="gray", vmin=lo, vmax=hi)
            ax.set_title(f"{image.shape[0]}×{image.shape[1]}", fontsize=10)
        except Exception as exc:
            ax.text(.5, .5, f"Decode failed\n{type(exc).__name__}", ha="center", va="center")
        ax.axis("off")
    for ax in axes.flat[len(pixel_files):]: ax.axis("off")
    fig.suptitle("Representative middle slices (1st–99th percentile window)")
    plt.tight_layout()
    plt.show()

## 9. Automated findings and modeling implications

In [ ]:
findings = []
least_covered = label_summary["coverage_pct"].idxmin()
rarest = label_summary["prevalence_pct"].idxmin()
most_common = label_summary["prevalence_pct"].idxmax()
findings.append(f"**Partial labels:** coverage ranges from {label_summary.coverage_pct.min():.1f}% to {label_summary.coverage_pct.max():.1f}%; {least_covered} has the least supervision.")
findings.append(f"**Class imbalance:** {rarest} is rarest ({label_summary.loc[rarest, 'prevalence_pct']:.1f}% positive) while {most_common} is most common ({label_summary.loc[most_common, 'prevalence_pct']:.1f}%).")
findings.append(f"**Reports:** {(reports.str.strip() != '').mean()*100:.1f}% of studies include non-empty report text; report availability and language/script variation should be preserved in validation.")
findings.append(f"**Series:** the median study has {series_per_study.median():.0f} series (range {series_per_study.min()}–{series_per_study.max()}); models must handle variable-sized inputs.")
if "all_three_planes" in plane_presence:
    findings.append(f"**Planes:** {plane_presence['all_three_planes'].mean()*100:.1f}% of studies contain sagittal, coronal, and axial series.")
display(Markdown("\n".join(f"- {item}" for item in findings)))

### Recommended modeling decisions

1. **Split at the study level.** All series and report text from a study must remain in one fold. If patient identifiers become available through allowed metadata, group by patient as well.
2. **Use iterative multilabel stratification where feasible.** Preserve both prevalence and label-availability patterns across folds; report per-target AUC and macro AUC.
3. **Mask missing labels.** Never fill blank targets with zero. Compute binary cross-entropy only for observed study–target pairs.
4. **Address imbalance per target.** Compare positive weighting, balanced sampling, and focal/asymmetric losses. Select using out-of-fold macro AUC, not accuracy.
5. **Normalize DICOMs robustly.** Apply modality/VOI transforms when present, correct MONOCHROME1 inversion, percentile-window intensities, standardize orientation and spacing, and sample slices consistently.
6. **Model variable protocols explicitly.** Aggregate within series, then across series/planes. Include sequence flags and plane as metadata; use masks for absent planes or sequences.
7. **Treat reports as privileged multimodal data.** Normalize Unicode and support multilingual text. Audit negation and explicit diagnosis terms. Compare image-only, text-only, and fused out-of-fold baselines to quantify leakage-like shortcuts and genuine complementarity.
8. **Avoid test-distribution assumptions.** The organizers warn that prevalence may shift. Prefer rank-robust validation and consider calibration only after the classifier is stable.
9. **Cache compact derivatives.** Precompute series manifests, selected slices, and normalized arrays to keep experiments within Kaggle's nine-hour notebook limit.

### Suggested next experiment

Build three leakage-safe baselines on identical folds: label-prior, TF-IDF report model, and a small 2.5D image model. Their per-target out-of-fold AUCs will reveal where text, images, and fusion contribute most before committing to a large architecture.